# Toto-2-4m-recipe tokenizer ablation → GIFT-Eval → HF Hub · **v10**

**Question:** which *context tokenizer* should the full pretrain use? Horizon decoding is fixed patch-32 + 9-quantile head in every arm; only how history is compressed into tokens varies.

| arm | tokenizer | history | ctx tokens | tests |
|---|---|---|---|---|
| **T0** | fixed-32 (Toto-2 control) | 4,096 | 128 | baseline |
| **T1** | pyramid, iso-context | 4,096 | 44 | H1: compression tax ≤ 1–2%? |
| **T2** | pyramid, iso-token | 16,384 | 128 | H2: long history helps at equal cost? |
| **T3** | adaptive equal-surprise | 16,384 | ~128 | does content-adaptive allocation beat a fixed schedule? |

**Recipe (Toto-2-4m clone, ~3.6M params, identical across arms):** decoder-only patched transformer, causal time attention with **index RoPE at time-scaled positions** (`pos = patch_start_time / 32` — identical to token index for T0), variate attention in the **last** layer, **CPM** training (contiguous patch masking in the fine band + masked tail), pinball loss in arcsinh-robust-scaled space, **NorMuon + AdamW** split, WSD schedule. Data: **TempoPFN-style prior ensemble** (native vectorized implementation of the same prior families; official generator pluggable via `OFFICIAL_SAMPLER`).

**Run order (important):**
1. Run the **SETUP** cell. It installs the GIFT-Eval stack (numpy<2) and **halts on purpose** the first time.
2. **Runtime → Restart session.**
3. **Run all.** Quick GIFT + HF-token checks run *before* training; the corpus builds to Drive (resumable); the 12-run matrix trains/evals/pushes one run at a time (resumable — reruns skip completed work).

**Budget:** ~2 A100-h per run at defaults (batch 64 × 8 variates) ≈ **~1 A100-day** for the matrix. Tip: set `SMOKE_MODE=True` in the RUN MATRIX cell first — it exercises the entire pipeline (train→eval→push) in minutes.

**Outputs:** Drive checkpoints under `tsfm_ckpts/v10_runs/`, models + metrics + model card on the HF Hub under `<you>/tsfm-toto2-4m-tokenizer-ablation`.

## 1 · Setup
Installs halt on first run (numpy downgrade) — restart, then Run all.

In [ ]:
# ============================================================================
# v10 SETUP  ·  RUN THIS CELL FIRST, THEN: Runtime > Restart session, THEN Run all.
# ----------------------------------------------------------------------------
# Installs the GIFT-Eval stack (gluonts 0.15 pins numpy<2 -> downgrade takes
# effect only after a restart, so this cell HALTS on purpose the first time)
# plus huggingface_hub for model uploads. Mounts Drive for checkpoints, the
# synthetic corpus, and the GIFT-Eval dataset cache.
# ============================================================================
import subprocess, sys, importlib.util, os

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

def _have(mod):
    return importlib.util.find_spec(mod) is not None

try:
    import torch
except ImportError:
    _pip("torch")

_pip("scipy", "pyarrow", "huggingface_hub")
if not (_have("gift_eval") and _have("gluonts")):
    _pip("git+https://github.com/SalesforceAIResearch/gift-eval.git",
         "gluonts", "orjson", "numpy<2")

# OPTIONAL: official TempoPFN generators (heavy deps; the notebook ships a
# native vectorized ensemble covering the same prior families, so this is
# not required). If you install it, wire it up via OFFICIAL_SAMPLER below.
# _pip("git+https://github.com/automl/TempoPFN.git")

# --- storage: /data mount > Colab Drive > local dir ---------------------------
if os.path.isdir("/data"):
    CKPT_DIR = "/data/tsfm_ckpts"                     # checkpoints, corpus, logs, results
    GIFT_EVAL_DIR = "/data/gifteval_data"             # GIFT-Eval dataset cache
    print("Using /data mount for all persistence.")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CKPT_DIR = "/content/drive/MyDrive/tsfm_ckpts"
        GIFT_EVAL_DIR = "/content/drive/MyDrive/gifteval_data"
    except Exception:
        CKPT_DIR = os.path.abspath("tsfm_ckpts")
        GIFT_EVAL_DIR = os.path.abspath("gifteval_data")
        print("No /data or Colab — persisting on local disk next to the notebook.")

# --- Weights & Biases (optional training telemetry) ---------------------------
# Auth once on the box: `wandb login` (or export WANDB_API_KEY=...). Headless
# without auth? export WANDB_MODE=offline and `wandb sync` later. If wandb is
# missing or unauthenticated the notebook prints a note and runs fine without it.
WANDB_ENABLE = True
WANDB_PROJECT = "tsfm-tokenizer-ablation-v10"
_pip("wandb")
os.makedirs(CKPT_DIR, exist_ok=True)
print("CKPT_DIR      =", CKPT_DIR)
print("GIFT_EVAL_DIR =", GIFT_EVAL_DIR)

# --- tokens: HF_TOKEN used for BOTH the gated GIFT-Eval download AND pushing
# models to the Hub. Needs a WRITE token if you want uploads to work.
# Colab: add a secret named HF_TOKEN. Elsewhere: export HF_TOKEN=hf_xxx.
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass
# os.environ["HF_TOKEN"] = "hf_xxx"       # or paste here (don't commit it)
HF_PUSH = True                            # set False to disable Hub uploads
HF_REPO = None                            # None -> "<username>/tsfm-toto2-4m-tokenizer-ablation"

# --- numpy ABI gate -----------------------------------------------------------
import numpy as np
print("numpy:", np.__version__)
if np.__version__.startswith("2"):
    raise RuntimeError(
        "numpy is still 2.x in this live kernel. The install downgraded it on disk.\n"
        ">>> Runtime > Restart session, then Run all. <<<  (Expected on the first run.)")

import math, time, json, random, warnings
from dataclasses import dataclass, field, asdict
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0),
          "| bf16:", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no GPU. The 12-run matrix needs ~1 A100-day; on CPU this "
          "notebook is only useful in smoke-test mode (see SMOKE_MODE below).")


## 1b · Quick GIFT-Eval + HF-token wiring test
Seasonal-naive on one dataset + Hub write-access check. If this PASSes, the long run is safe.

In [ ]:
# ============================================================================
# QUICK GIFT-Eval WIRING TEST  ·  runs BEFORE training so env problems surface
# now, not after an A100-day. Trivial seasonal-naive predictor — no model.
# Downloads ONLY one small dataset (m4_weekly). Also sanity-checks the HF
# token so Hub pushes at the end of each run won't silently fail.
# ============================================================================
QUICK_TEST = True
if QUICK_TEST:
    import os, numpy as np
    assert np.__version__.startswith("1."), \
        "numpy is 2.x — Runtime > Restart session, then Run all."

    from huggingface_hub import snapshot_download
    from scipy.stats import norm
    from gluonts.model import evaluate_model
    from gluonts.model.forecast import QuantileForecast
    from gluonts.model.predictor import RepresentablePredictor
    from gluonts.time_feature import get_seasonality
    from gluonts.ev.metrics import MASE, MeanWeightedSumQuantileLoss
    from gift_eval.data import Dataset as GEDataset

    QL = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    _name, _term = "m4_weekly", "short"

    if not os.path.exists(os.path.join(GIFT_EVAL_DIR, _name)):
        print(f"[quick] downloading only {_name} (first run)...")
        snapshot_download("Salesforce/GiftEval", repo_type="dataset",
                          local_dir=GIFT_EVAL_DIR, token=os.environ.get("HF_TOKEN") or None,
                          allow_patterns=[f"{_name}/*", f"{_name}*"])
    os.environ["GIFT_EVAL"] = GIFT_EVAL_DIR

    class _SNaive(RepresentablePredictor):
        def __init__(self, H, m):
            super().__init__(prediction_length=H); self.m = max(1, m)
        def predict(self, dataset, **kwargs):
            H, m = self.prediction_length, self.m
            for e in dataset:
                y = np.asarray(e["target"], np.float32)
                base = (np.tile(y[-m:], int(np.ceil(H/m)))[:H] if len(y) >= m
                        else np.full(H, y[-1] if len(y) else 0.0))
                r = float(np.std(np.diff(y))) if len(y) > 1 else 1.0
                q = np.stack([base + norm.ppf(p)*r for p in QL], axis=0)
                yield QuantileForecast(
                    forecast_arrays=q, forecast_keys=[str(p) for p in QL],
                    start_date=e["start"] + len(e["target"]), item_id=e.get("item_id"))

    try:
        to_uni = GEDataset(name=_name, term=_term).target_dim != 1
        ds = GEDataset(name=_name, term=_term, to_univariate=to_uni)
        season = get_seasonality(ds.freq)
        res = evaluate_model(
            _SNaive(ds.prediction_length, season), test_data=ds.test_data,
            metrics=[MASE(), MeanWeightedSumQuantileLoss(quantile_levels=QL)],
            batch_size=512, axis=None, mask_invalid_label=True,
            allow_nan_forecast=False, seasonality=season)
        print(f"PASS  {_name}/{_term}  freq={ds.freq}  H={ds.prediction_length}  "
              f"MASE={float(res['MASE[0.5]'].iloc[0]):.3f}  "
              f"CRPS={float(res['mean_weighted_sum_quantile_loss'].iloc[0]):.3f}")
        print(">>> GIFT-Eval data + eval path verified. Safe to train below. <<<")
    except KeyError:
        print("FAIL: HF_TOKEN missing, or license not accepted at "
              "huggingface.co/datasets/Salesforce/GiftEval")
    except Exception as ex:
        print(f"FAIL: {type(ex).__name__}: {ex}")

    # --- HF write-access check (for model pushes) -----------------------------
    if HF_PUSH:
        try:
            from huggingface_hub import HfApi
            info = HfApi(token=os.environ.get("HF_TOKEN") or None).whoami()
            perm = (info.get("auth", {}).get("accessToken", {}) or {}).get("role", "?")
            print(f"HF login OK as '{info['name']}' (token role: {perm}) — "
                  f"models will push to {HF_REPO or info['name'] + '/tsfm-toto2-4m-tokenizer-ablation'}")
            if perm == "read":
                print("WARNING: token is READ-only — model uploads will fail. "
                      "Use a WRITE token if you want Hub saving.")
        except Exception as ex:
            print(f"HF login check failed ({type(ex).__name__}) — pushes will be skipped.")


## 2 · Synthetic prior ensemble + corpus
Native vectorized implementations of the TempoPFN prior families (GP priors via spectral/RFF approximation). Corpus builds once to Drive, sharded + resumable; training samples deterministic paired crops with augmentation.

In [ ]:
# ============================================================================
# ENSEMBLE PFN SYNTHETIC GENERATOR  ·  v10
# TempoPFN-style prior ensemble (Moroshan et al. 2025), honestly labeled:
# these are OUR vectorized implementations of the same prior FAMILIES
# (KernelSynth / GP / SDE / sawtooth / step / spikes / sine / ETS / fractal),
# not the official automl/TempoPFN code. GP-family priors use a spectral
# (random-Fourier-feature) approximation instead of O(n^3) Cholesky so that
# 16k-step series are cheap. An optional cell below can swap in the official
# TempoPFN package via set_corpus() if you install it.
# All generators: (B, L, rng) -> float32 [B, L], vectorized over B.
# ============================================================================
import numpy as np

def _envelope(B, L, rng, smooth=None):
    """Smooth positive amplitude envelope via low-freq cosines."""
    t = np.linspace(0, 1, L)[None, :]
    f = rng.uniform(0.5, 3.0, (B, 1))
    ph = rng.uniform(0, 2*np.pi, (B, 1))
    e = 1.0 + rng.uniform(0.1, 0.8, (B, 1)) * np.cos(2*np.pi*f*t + ph)
    return np.clip(e, 0.05, None)

def gen_kernelsynth_spectral(B, L, rng):
    """KernelSynth-family (Chronos): composite kernels ~ sum of periodic
    (ExpSineSquared -> harmonic stacks), stationary (RBF/RQ -> RFF), noise.
    Spectral approximation: exact for kernel ADDITION; kernel products are
    approximated by amplitude-modulating periodic parts with a smooth envelope."""
    t = np.arange(L, dtype=np.float64)[None, :]
    x = np.zeros((B, L))
    # periodic components: 1-3 base periods, few harmonics each
    for _ in range(int(rng.integers(1, 4))):
        p = np.exp(rng.uniform(np.log(6), np.log(L))) * np.ones((B, 1))
        p *= np.exp(rng.uniform(-0.1, 0.1, (B, 1)))          # jitter per series
        comp = np.zeros((B, L))
        for h in range(1, 4):
            a = rng.uniform(0, 1, (B, 1)) / h
            ph = rng.uniform(0, 2*np.pi, (B, 1))
            comp += a * np.sin(2*np.pi*h*t/p + ph)
        if rng.random() < 0.5:                                # ~ periodic*RBF product
            comp *= _envelope(B, L, rng)
        x += rng.uniform(0.3, 1.5, (B, 1)) * comp
    # stationary RBF/RQ via random Fourier features
    if rng.random() < 0.8:
        F = 64
        ls = np.exp(rng.uniform(np.log(L/64), np.log(L/2), (B, 1, 1)))
        w = rng.standard_normal((B, F, 1)) / ls               # RBF spectral: N(0, 1/ls^2)
        if rng.random() < 0.4:                                # RQ = gamma mixture of RBF
            g = rng.gamma(rng.uniform(1, 5), 1.0, (B, F, 1))
            w = w * np.sqrt(g / np.maximum(g.mean(axis=1, keepdims=True), 1e-9))
        ph = rng.uniform(0, 2*np.pi, (B, F, 1))
        a = rng.standard_normal((B, F, 1)) * np.sqrt(2.0/F)
        x += rng.uniform(0.3, 1.2, (B, 1)) * (a*np.cos(w*t[:, None, :] + ph)).sum(1)
    x += rng.uniform(0.0, 0.15, (B, 1)) * rng.standard_normal((B, L))   # WhiteKernel
    return x.astype(np.float32)

def gen_gp_spectral(B, L, rng):
    """GP-family (Mamba4Cast-style): up to 6 components incl. Matern (spectral
    density ~ Student-t -> sample freqs from t-dist), linear, poly-trend."""
    t = np.arange(L, dtype=np.float64)[None, :]
    x = np.zeros((B, L))
    for _ in range(int(rng.integers(2, 7))):
        kind = rng.choice(["matern", "rbf", "periodic", "linear", "poly"])
        if kind in ("matern", "rbf"):
            F = 48
            ls = np.exp(rng.uniform(np.log(L/128), np.log(L), (B, 1, 1)))
            if kind == "matern":
                nu = rng.choice([1.5, 2.5])
                w = rng.standard_t(2*nu, (B, F, 1)) / ls
            else:
                w = rng.standard_normal((B, F, 1)) / ls
            ph = rng.uniform(0, 2*np.pi, (B, F, 1))
            a = rng.standard_normal((B, F, 1)) * np.sqrt(2.0/F)
            x += rng.uniform(0.2, 1.0, (B, 1)) * (a*np.cos(w*t[:, None, :] + ph)).sum(1)
        elif kind == "periodic":
            p = np.exp(rng.uniform(np.log(8), np.log(L/2))) * np.exp(rng.uniform(-0.1, 0.1, (B, 1)))
            x += rng.uniform(0.2, 1.2, (B, 1)) * np.sin(2*np.pi*t/p + rng.uniform(0, 2*np.pi, (B, 1)))
        elif kind == "linear":
            x += rng.normal(0, 1.0, (B, 1)) * (t/L - rng.uniform(0, 1, (B, 1)))
        else:
            c = rng.normal(0, 0.5, (B, 3))
            u = t/L
            x += c[:, :1]*u + c[:, 1:2]*u**2 + c[:, 2:3]*u**3
    return x.astype(np.float32)

def gen_sde_ou(B, L, rng):
    """Novel-prior family: regime-switching Ornstein-Uhlenbeck (vectorized Euler).
    theta/mu/sigma jump at 0-5 changepoints per series."""
    n_reg = 1 + rng.integers(0, 6, B)
    theta = np.exp(rng.uniform(np.log(1e-3), np.log(0.3), (B, 6)))
    mu    = rng.normal(0, 1.5, (B, 6))
    sig   = np.exp(rng.uniform(np.log(0.02), np.log(0.6), (B, 6)))
    # regime index per timestep
    reg = np.zeros((B, L), dtype=np.int64)
    for b in range(B):                                # cheap loop: only builds indices
        cps = np.sort(rng.choice(np.arange(1, L), size=n_reg[b]-1, replace=False)) if n_reg[b] > 1 else []
        prev, r = 0, 0
        for cp in list(cps) + [L]:
            reg[b, prev:cp] = r; prev, r = cp, r+1
    th = np.take_along_axis(theta, reg, 1); m = np.take_along_axis(mu, reg, 1)
    s  = np.take_along_axis(sig, reg, 1)
    e = rng.standard_normal((B, L))
    x = np.zeros((B, L)); prev = rng.normal(0, 1, B)
    for i in range(L):                                # O(L) scan, vectorized over B
        prev = prev + th[:, i]*(m[:, i]-prev) + s[:, i]*e[:, i]
        x[:, i] = prev
    return x.astype(np.float32)

def gen_sawtooth(B, L, rng):
    t = np.arange(L)[None, :]
    p = np.exp(rng.uniform(np.log(8), np.log(L/2), (B, 1)))
    ph = rng.uniform(0, 1, (B, 1))
    frac = ((t/p + ph) % 1.0)
    skew = rng.uniform(0.05, 0.95, (B, 1))            # temporal asymmetry
    x = np.where(frac < skew, frac/skew, (1-frac)/(1-skew)) * rng.uniform(0.5, 2, (B, 1))
    return (x - x.mean(1, keepdims=True)).astype(np.float32)

def gen_step(B, L, rng):
    x = np.zeros((B, L))
    for b in range(B):
        n = rng.integers(1, 8)
        cps = np.sort(rng.choice(np.arange(1, L), n, replace=False))
        lv, prev = 0.0, 0
        for cp in list(cps) + [L]:
            x[b, prev:cp] = lv; lv += rng.normal(0, 1.0); prev = cp
    return x.astype(np.float32)

def gen_spikes(B, L, rng):
    base = 0.05*rng.standard_normal((B, L))
    m = rng.random((B, L)) < rng.uniform(0.005, 0.05, (B, 1))
    sgn = np.where(rng.random((B, 1)) < 0.8, 1.0, -1.0)   # mostly upper-tail (latency-like)
    return (base + m * sgn * rng.exponential(rng.uniform(0.5, 3, (B, 1)), (B, L))).astype(np.float32)

def gen_sine(B, L, rng):
    t = np.arange(L)[None, :]
    p = np.exp(rng.uniform(np.log(6), np.log(L), (B, 1)))
    return (rng.uniform(0.5, 2, (B, 1)) * np.sin(2*np.pi*t/p + rng.uniform(0, 2*np.pi, (B, 1)))).astype(np.float32)

def gen_ets(B, L, rng):
    """ForecastPFN-family: multiplicative Error-Trend-Seasonality."""
    t = np.arange(L)[None, :] / L
    trend = 1.0 + rng.normal(0, 0.5, (B, 1))*t + rng.normal(0, 0.3, (B, 1))*t**2
    seas = np.ones((B, L))
    for p0 in (7, 24, 168, 12):
        if rng.random() < 0.5:
            seas *= 1.0 + rng.uniform(0, 0.4, (B, 1))*np.sin(2*np.pi*np.arange(L)[None, :]/p0 + rng.uniform(0, 2*np.pi, (B, 1)))
    err = 1.0 + rng.uniform(0.01, 0.1, (B, 1))*rng.standard_normal((B, L))
    x = trend * np.clip(seas, 0.05, None) * err
    return (x - x.mean(1, keepdims=True)).astype(np.float32)

def gen_fractal(B, L, rng):
    """Audio-inspired multi-scale fractal: 1/f^beta spectral synthesis (rfft)."""
    beta = rng.uniform(0.5, 2.5, (B, 1))
    n_f = L//2 + 1
    f = np.arange(1, n_f)[None, :]
    amp = f ** (-beta/2)
    ph = rng.uniform(0, 2*np.pi, (B, n_f-1))
    spec = np.zeros((B, n_f), dtype=np.complex128)
    spec[:, 1:] = amp * np.exp(1j*ph)
    x = np.fft.irfft(spec, n=L, axis=1)
    x = x / (x.std(1, keepdims=True) + 1e-9)
    return x.astype(np.float32)

def gen_anomaly(B, L, rng):
    """Base signal + injected anomaly windows (level/variance bursts)."""
    x = gen_sine(B, L, rng) + 0.1*rng.standard_normal((B, L)).astype(np.float32)
    for b in range(B):
        for _ in range(rng.integers(1, 4)):
            s = rng.integers(0, max(1, L-20)); w = rng.integers(5, max(6, L//20))
            if rng.random() < 0.5: x[b, s:s+w] += rng.normal(0, 2.0)
            else:                  x[b, s:s+w] *= rng.uniform(2, 5)
    return x

# --------------------------------------------------------------------------- ensemble
GEN_WEIGHTS = {          # roughly following TempoPFN's family coverage; adjust freely
    gen_kernelsynth_spectral: 0.20, gen_gp_spectral: 0.20, gen_sde_ou: 0.12,
    gen_ets: 0.12, gen_fractal: 0.10, gen_sawtooth: 0.07, gen_step: 0.07,
    gen_sine: 0.05, gen_spikes: 0.04, gen_anomaly: 0.03,
}

def _postprocess(x, rng):
    """Random compositions + global affine, mirroring PFN prior richness."""
    B, L = x.shape
    if rng.random() < 0.3:  x += gen_step(B, L, rng) * 0.7                   # level shifts
    if rng.random() < 0.3:  x += gen_spikes(B, L, rng) * 0.7                 # rare spikes
    if rng.random() < 0.5:  x += rng.normal(0, 1.0/L, (B, 1)) * np.arange(L)[None, :]  # drift
    if rng.random() < 0.15: x = np.exp(np.clip(0.5*x, -6, 6)) - 1.0          # positivity/skew
    x = x * np.exp(rng.uniform(np.log(0.2), np.log(5.0), (B, 1))) + rng.normal(0, 2.0, (B, 1))
    return x.astype(np.float32)

def sample_ensemble(B, L, rng):
    """Draw B series of length L from the mixed prior ensemble."""
    gens = list(GEN_WEIGHTS.keys())
    w = np.array(list(GEN_WEIGHTS.values())); w = w / w.sum()
    counts = rng.multinomial(B, w)
    outs = [g(int(c), L, rng) for g, c in zip(gens, counts) if c > 0]
    x = np.concatenate(outs, 0)
    return _postprocess(x, rng)[rng.permutation(B)]

# --- optional: official TempoPFN generators --------------------------------
# To use the official automl/TempoPFN prior instead of the native ensemble:
#   !pip install "git+https://github.com/automl/TempoPFN.git"
# then adapt their generator wrappers (src/synthetic_generation/) to a callable
# (B, L, rng) -> [B, L] and assign it:  OFFICIAL_SAMPLER = my_fn
# The corpus builder below will prefer OFFICIAL_SAMPLER when it is not None.
OFFICIAL_SAMPLER = None
print("ensemble generator ready:", [g.__name__ for g in GEN_WEIGHTS])


In [ ]:
# ============================================================================
# CORPUS  ·  pre-generate the synthetic corpus to Drive (resumable, sharded)
# ----------------------------------------------------------------------------
# Two pools:
#   short : len = 4096 + 1024   -> arms T0/T1  (ctx 4096  + train tail)
#   long  : len = 16384 + 1024  -> arms T2/T3  (ctx 16384 + train tail)
# Training samples random crops + affine/noise augmentation, so heavy corpus
# reuse is fine (TempoPFN itself reuses a 10M-series corpus for 4M iters).
# Paired comparisons: arms sharing a pool + data_seed see IDENTICAL batches.
# ============================================================================
import os, numpy as np

CORPUS_DIR = os.path.join(CKPT_DIR, "corpus_v10")
os.makedirs(CORPUS_DIR, exist_ok=True)

SHORT_LEN, LONG_LEN = 4096 + 1024, 16384 + 1024
N_SHORT, N_LONG     = 40_000, 10_000          # ~0.8 GB + ~0.7 GB float32 on Drive
SHARD               = 2_000

def build_pool(name, n_series, length, seed0=1234):
    n_shards = (n_series + SHARD - 1) // SHARD
    sampler = OFFICIAL_SAMPLER or sample_ensemble
    for si in range(n_shards):
        path = os.path.join(CORPUS_DIR, f"{name}_{si:04d}.npy")
        if os.path.exists(path):
            continue
        rng = np.random.default_rng(seed0 + si)
        n = min(SHARD, n_series - si*SHARD)
        x = sampler(n, length, rng)
        np.save(path, x.astype(np.float32))
        print(f"[corpus] {name} shard {si+1}/{n_shards} written ({n}x{length})")
    print(f"[corpus] {name}: complete ({n_series} series x {length})")

BUILD_CORPUS = True
if BUILD_CORPUS:
    build_pool("short", N_SHORT, SHORT_LEN, seed0=1000)
    build_pool("long",  N_LONG,  LONG_LEN,  seed0=9000)

class CorpusSampler:
    """Memory-maps corpus shards; deterministic paired sampling per (data_seed, step)."""
    def __init__(self, name, data_seed):
        import glob
        paths = sorted(glob.glob(os.path.join(CORPUS_DIR, f"{name}_*.npy")))
        assert paths, f"no corpus shards for pool '{name}' — run the corpus builder"
        self.shards = [np.load(p, mmap_mode="r") for p in paths]
        self.counts = np.array([s.shape[0] for s in self.shards])
        self.offsets = np.concatenate([[0], np.cumsum(self.counts)])
        self.total = int(self.counts.sum())
        self.length = self.shards[0].shape[1]
        self.data_seed = data_seed

    def batch(self, step, n, crop_len):
        """[n, crop_len] float32, deterministic in (data_seed, step)."""
        rng = np.random.default_rng((self.data_seed * 1_000_003 + step) % (2**63))
        idx = rng.integers(0, self.total, n)
        starts = rng.integers(0, self.length - crop_len + 1, n)
        out = np.empty((n, crop_len), np.float32)
        for i, (j, s) in enumerate(zip(idx, starts)):
            si = int(np.searchsorted(self.offsets, j, side="right") - 1)
            out[i] = self.shards[si][j - self.offsets[si], s:s+crop_len]
        # augmentation: sign flip, log-scale, offset, small noise
        sgn = np.where(rng.random((n, 1)) < 0.2, -1.0, 1.0)
        a = sgn * np.exp(rng.uniform(np.log(0.5), np.log(2.0), (n, 1)))
        b = rng.normal(0, 1.0, (n, 1)) * (np.abs(out).mean(1, keepdims=True) + 1e-3)
        out = a*out + b + rng.normal(0, 0.01, (n, 1))*out.std(1, keepdims=True)*rng.standard_normal(out.shape)
        return out.astype(np.float32)

def sample_series(sampler, step, cfg):
    """[B, V, ctx_span + kmax*patch] — variates drawn independently (see notes)."""
    L = cfg.ctx_span + cfg.train_kmax * cfg.patch
    flat = sampler.batch(step, cfg.batch * cfg.n_variates, L)
    return flat.reshape(cfg.batch, cfg.n_variates, L)

print("corpus ready:", CORPUS_DIR)


## 3 · Config, tokenizers, model, optimizer
The three tokenizers share one feature contract (pooled values ++ pooled mask ++ log seg-length) so the embedding is identical and **all arms have exactly the same parameter count** — no capacity confound.

In [ ]:
# ============================================================================
# CONFIG  ·  Toto-2-4m-recipe clone + tokenizer switch + run matrix
# 4m recipe anchors: d=256, 4 heads, 4 layers, patch 32, ctx 4096, quantile
# head (9 levels, pinball), CPM (c_max=16, p_max<=0.4), NorMuon+AdamW split,
# arcsinh robust scaling, variate attention in the LAST layer, index RoPE.
# Deviations are flagged inline (V=8 default vs paper's 32 for single-GPU;
# CPM restricted to the fine band + tail so arms stay comparable; no u-muP,
# so LRs are re-calibrated once on T0 and shared).
# ============================================================================
import math, json, os
from dataclasses import dataclass, field, asdict

STD_LEVELS_LIST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

@dataclass
class Cfg:
    run_name: str = "T0_fixed"
    # --- tokenizer ---------------------------------------------------------
    tokenizer: str = "fixed"          # fixed | pyramid | adaptive
    patch: int = 32                   # fine/inner patch (matches Toto-2 P=32)
    ctx_span: int = 4096              # raw history steps consumed
    pyramid_levels: tuple = ()        # ((span, plen), ...) oldest->newest; last plen == patch
    adaptive_hist_tokens: int = 96    # equal-surprise segments over pre-fine history
    fine_span: int = 1024             # fine band (patch-32 tokens) = CPM zone
    # --- model (Toto-2 4m) --------------------------------------------------
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    # --- CPM ----------------------------------------------------------------
    train_kmin: int = 4               # tail-mask length in patches
    train_kmax: int = 32
    cpm_cmax: int = 16                # max interior contiguous spans
    cpm_pmax: float = 0.4             # max masked fraction of (zone + tail) tokens
    cpm_span_max: int = 4             # max interior span length (patches)
    # --- train ---------------------------------------------------------------
    steps: int = 30_000
    batch: int = 64
    n_variates: int = 8               # paper: 32 — raise if your GPU allows
    lr: float = 1e-3                  # AdamW group (embed/head/bias/norm)
    normuon_lr: float = 8e-4          # NorMuon group (transformer matrices)
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    warmup_frac: float = 0.03
    decay_frac: float = 0.10          # WSD: linear decay tail
    seed: int = 0                     # model init seed
    data_seed: int = 0                # corpus stream seed (paired across arms)
    pool: str = "short"               # short | long corpus pool
    # --- eval / io -----------------------------------------------------------
    eval_every: int = 500
    ckpt_every: int = 2_500
    gift_ckpts: tuple = (15_000, 30_000)

    @property
    def zone_tokens(self):  return self.fine_span // self.patch
    @property
    def n_ctx_tokens(self):
        if self.tokenizer == "fixed":    return self.ctx_span // self.patch
        if self.tokenizer == "pyramid":  return sum(s//p for s, p in self.pyramid_levels)
        return self.adaptive_hist_tokens + self.zone_tokens

# ---------------------------------------------------------------------------- arm presets
def make_arm(name, seed):
    base = dict(seed=seed, data_seed=seed)
    if name == "T0":   # fixed-32 control — 4096 steps / 128 tokens
        return Cfg(run_name=f"T0_fixed_s{seed}", tokenizer="fixed",
                   ctx_span=4096, pool="short", **base)
    if name == "T1":   # pyramid iso-context — 4096 steps / 44 tokens
        return Cfg(run_name=f"T1_pyr_ctx_s{seed}", tokenizer="pyramid", ctx_span=4096,
                   pyramid_levels=((2048, 512), (1024, 128), (1024, 32)),
                   fine_span=1024, pool="short", **base)
    if name == "T2":   # pyramid iso-token — 16384 steps / 128 tokens
        return Cfg(run_name=f"T2_pyr_tok_s{seed}", tokenizer="pyramid", ctx_span=16384,
                   pyramid_levels=((8192, 512), (4096, 128), (3072, 64), (1024, 32)),
                   fine_span=1024, pool="long", **base)
    if name == "T3":   # adaptive equal-surprise — 16384 steps / ~128 tokens
        return Cfg(run_name=f"T3_adaptive_s{seed}", tokenizer="adaptive", ctx_span=16384,
                   adaptive_hist_tokens=96, fine_span=1024, pool="long", **base)
    raise ValueError(name)

SEEDS = (0, 1, 2)
RUNS = [make_arm(a, s) for a in ("T0", "T1", "T2", "T3") for s in SEEDS]
for c in RUNS:
    print(f"{c.run_name:18s} ctx={c.ctx_span:6d} tokens={c.n_ctx_tokens:4d} pool={c.pool}")


In [ ]:
# ============================================================================
# TOKENIZERS  ·  fixed | pyramid | adaptive   (+ mask-aware scaler, RoPE)
# ----------------------------------------------------------------------------
# Shared contract: tokenize(z, m) -> (feat [BV, N, 2*inner+1], pos [N] or [BV,N])
#   z    : arcsinh-scaled series [BV, L] with masked steps zeroed
#   m    : per-timestep mask channel [BV, L] (1 = masked / to-predict)
#   feat : pooled values (inner=patch) ++ pooled mask ++ log(seg_len/patch)
#   pos  : TIME-SCALED RoPE positions = patch_start_time / patch. Identical to
#          token index for the fixed tokenizer (Toto default); preserves
#          temporal geometry for non-uniform patches.
# Only the fine band (last fine_span steps, patch-size `patch`) + appended
# tail patches are ever masked or decoded — the head stays fixed patch-32.
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F
import math

class RobustScaler:
    """Causal arcsinh scaler; mask-aware (masked steps excluded via nanmedian)."""
    def __init__(self, x, m=None):
        xx = x.clone()
        if m is not None:
            xx[m > 0.5] = float("nan")
        self.loc = xx.nanmedian(dim=-1, keepdim=True).values
        self.loc = torch.nan_to_num(self.loc, nan=0.0)
        self.scale = (xx - self.loc).abs().nanmedian(dim=-1, keepdim=True).values
        self.scale = torch.nan_to_num(self.scale, nan=1.0).clamp_min(1e-3)
    def encode(self, x): return torch.asinh((x - self.loc) / self.scale)
    def decode(self, z): return torch.sinh(z) * self.scale + self.loc

def rope_angles(positions, dim, base=10000.0):
    """positions [N] or [BV,N] (float ok) -> angles [.., N, dim/2]."""
    i = torch.arange(0, dim, 2, device=positions.device).float()
    inv = base ** (-i / dim)
    return positions.float().unsqueeze(-1) * inv

def apply_rope(x, ang):
    """x [B,H,N,Dh]; ang [N,Dh/2] or [B,N,Dh/2]."""
    if ang.dim() == 2:  ang = ang[None, None]          # [1,1,N,d2]
    else:               ang = ang[:, None]             # [B,1,N,d2]
    cos = torch.cos(ang).repeat_interleave(2, dim=-1)
    sin = torch.sin(ang).repeat_interleave(2, dim=-1)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    rot = torch.stack((-x2, x1), dim=-1).flatten(-2)
    return x * cos + rot * sin

def _pool_patch(seg, inner):
    """[BV, n, L] -> [BV, n, inner]: mean-pool (antialias) or repeat-upsample."""
    L = seg.shape[-1]
    if L == inner: return seg
    if L > inner:
        assert L % inner == 0
        return seg.reshape(*seg.shape[:-1], inner, L // inner).mean(-1)
    assert inner % L == 0
    return seg.repeat_interleave(inner // L, dim=-1)

def _feat(vals, msk, seg_len, inner):
    """concat pooled values, pooled mask, log(seg_len/inner) -> [BV, n, 2*inner+1]."""
    ll = torch.full((*vals.shape[:-1], 1), math.log(seg_len / inner),
                    device=vals.device, dtype=vals.dtype)
    return torch.cat([vals, msk, ll], dim=-1)

class FixedTokenizer(nn.Module):
    def __init__(self, cfg): super().__init__(); self.cfg = cfg
    def forward(self, z, m):
        P, BV = self.cfg.patch, z.shape[0]
        n = z.shape[1] // P
        v = z.reshape(BV, n, P); mm = m.reshape(BV, n, P)
        feat = _feat(v, mm, P, P)
        pos = torch.arange(n, device=z.device).float()          # == start/P
        return feat, pos

class PyramidTokenizer(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg
        self.levels = cfg.pyramid_levels
        assert sum(s for s, _ in self.levels) == cfg.ctx_span
        assert self.levels[-1][1] == cfg.patch and self.levels[-1][0] == cfg.fine_span
        # level identity is carried by the log(seg_len/patch) feature + time position
    def forward(self, z, m):
        P, BV = self.cfg.patch, z.shape[0]
        feats, poss, ofs = [], [], 0
        for span, plen in self.levels:
            n = span // plen
            v = _pool_patch(z[:, ofs:ofs+span].reshape(BV, n, plen), P)
            mm = _pool_patch(m[:, ofs:ofs+span].reshape(BV, n, plen), P)
            feats.append(_feat(v, mm, plen, P))
            poss.append((torch.arange(n, device=z.device).float() * plen + ofs) / P)
            ofs += span
        return torch.cat(feats, 1), torch.cat(poss)

class AdaptiveTokenizer(nn.Module):
    """Equal-surprise segmentation of the pre-fine history into a fixed token
    budget K (deterministic per series -> eval-stable), plus the fixed fine band.
    Surprise = smoothed |dz|; boundaries at surprise-CDF quantiles with a uniform
    floor mixed in (guarantees no degenerate segments); segments are pooled to
    `patch` inner values via linear-interp gather from a mildly smoothed copy."""
    def __init__(self, cfg, floor=0.35):
        super().__init__(); self.cfg = cfg; self.floor = floor
    def forward(self, z, m):
        cfg, P = self.cfg, self.cfg.patch
        BV = z.shape[0]
        Lh = cfg.ctx_span - cfg.fine_span                    # history region
        K = cfg.adaptive_hist_tokens
        zh, mh = z[:, :Lh], m[:, :Lh]
        # --- surprise + equal-surprise boundaries ---------------------------
        s = (zh[:, 1:] - zh[:, :-1]).abs()
        s = F.avg_pool1d(s[:, None], 9, 1, 4)[:, 0]
        s = F.pad(s, (1, 0), value=0.0)
        s = (1 - self.floor) * s / (s.sum(1, keepdim=True) + 1e-8) + self.floor / Lh
        cs = s.cumsum(1)                                     # [BV, Lh] in (0,1]
        tgt = (torch.arange(1, K, device=z.device).float() / K)[None].expand(BV, -1)
        bnd = torch.searchsorted(cs.contiguous(), tgt.contiguous())      # [BV, K-1]
        bnd = bnd.clamp(1, Lh - 1)
        bnd, _ = torch.cummax(bnd, dim=1)                    # enforce monotone
        starts = torch.cat([torch.zeros(BV, 1, device=z.device, dtype=bnd.dtype), bnd], 1)
        ends   = torch.cat([bnd, torch.full((BV, 1), Lh, device=z.device, dtype=bnd.dtype)], 1)
        seg_len = (ends - starts).clamp_min(1).float()       # [BV, K]
        # --- pooled inner values via interp gather (2 mip levels) ------------
        z_sm = F.avg_pool1d(zh[:, None], 17, 1, 8)[:, 0]     # antialiased copy
        use_sm = (seg_len > 2*P)[..., None]                  # long segs -> smoothed
        frac = (torch.arange(P, device=z.device).float() + 0.5) / P
        pos_f = starts[..., None].float() + seg_len[..., None] * frac    # [BV,K,P]
        i0 = pos_f.floor().long().clamp(0, Lh - 1); i1 = (i0 + 1).clamp(0, Lh - 1)
        w = (pos_f - i0.float())
        def gath(src): 
            g0 = torch.gather(src[:, None].expand(-1, K, -1), 2, i0)
            g1 = torch.gather(src[:, None].expand(-1, K, -1), 2, i1)
            return g0 * (1 - w) + g1 * w
        v = torch.where(use_sm, gath(z_sm), gath(zh))
        mm = gath(mh)                                        # history is never masked, ~0
        ll = torch.log(seg_len / P)[..., None]
        feat_h = torch.cat([v, mm, ll], dim=-1)              # [BV, K, 2P+1]
        pos_h = ((starts.float() + ends.float()) * 0.5) / P  # segment-center time / P
        # --- fine band (identical to fixed) ----------------------------------
        nf = cfg.zone_tokens
        vf = z[:, Lh:].reshape(BV, nf, P); mf = m[:, Lh:].reshape(BV, nf, P)
        feat_f = _feat(vf, mf, P, P)
        pos_f2 = ((torch.arange(nf, device=z.device).float() * P + Lh) / P)[None].expand(BV, -1)
        return torch.cat([feat_h, feat_f], 1), torch.cat([pos_h, pos_f2], 1)

def build_tokenizer(cfg):
    return {"fixed": FixedTokenizer, "pyramid": PyramidTokenizer,
            "adaptive": AdaptiveTokenizer}[cfg.tokenizer](cfg)


In [ ]:
# ============================================================================
# MODEL  ·  MiniTSFM2 — Toto-2-4m-recipe backbone with pluggable tokenizer
# decoder-only, causal time attention (index RoPE @ time-scaled positions),
# variate attention in the LAST layer, quantile head (9 levels, patch 32),
# CPM training (interior spans in the fine band + masked tail), pinball loss
# in z-space. Only fine tokens are decoded; coarse tokens are context-only.
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F
import math

QUANTILES = torch.tensor(STD_LEVELS_LIST)

class TimeAttn(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.dh = cfg.n_heads, cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3*cfg.d_model)
        self.o = nn.Linear(cfg.d_model, cfg.d_model)
    def forward(self, x, ang):
        B, N, D = x.shape
        q, k, v = self.qkv(x).chunk(3, -1)
        q, k, v = [t.view(B, N, self.h, self.dh).transpose(1, 2) for t in (q, k, v)]
        q, k = apply_rope(q, ang), apply_rope(k, ang)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(out.transpose(1, 2).reshape(B, N, D))

class VariateAttn(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.dh = cfg.n_heads, cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3*cfg.d_model)
        self.o = nn.Linear(cfg.d_model, cfg.d_model)
    def forward(self, x):                       # x [B*N, V, D]
        B, V, D = x.shape
        q, k, v = self.qkv(x).chunk(3, -1)
        q, k, v = [t.view(B, V, self.h, self.dh).transpose(1, 2) for t in (q, k, v)]
        out = F.scaled_dot_product_attention(q, k, v)
        return self.o(out.transpose(1, 2).reshape(B, V, D))

class Block(nn.Module):
    def __init__(self, cfg, use_variate):
        super().__init__()
        self.n1 = nn.LayerNorm(cfg.d_model); self.attn = TimeAttn(cfg)
        self.use_variate = use_variate
        if use_variate:
            self.nv = nn.LayerNorm(cfg.d_model); self.vattn = VariateAttn(cfg)
        self.n2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, 4*cfg.d_model), nn.SiLU(),
                                nn.Linear(4*cfg.d_model, cfg.d_model))
    def forward(self, x, ang, V, N):
        x = x + self.attn(self.n1(x), ang)
        if self.use_variate and V > 1:
            B = x.shape[0] // V
            xr = self.nv(x).view(B, V, N, -1).permute(0, 2, 1, 3).reshape(B*N, V, -1)
            mix = self.vattn(xr).view(B, N, V, -1).permute(0, 2, 1, 3).reshape(B*V, N, -1)
            x = x + mix
        return x + self.ff(self.n2(x))

class MiniTSFM2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        P = cfg.patch
        self.tokenizer = build_tokenizer(cfg)
        self.embed = nn.Sequential(nn.Linear(2*P + 1, cfg.d_model), nn.SiLU(),
                                   nn.Linear(cfg.d_model, cfg.d_model))
        self.mask_tok = nn.Parameter(torch.randn(cfg.d_model) * 0.02)
        self.blocks = nn.ModuleList([Block(cfg, use_variate=(l == cfg.n_layers - 1))
                                     for l in range(cfg.n_layers)])
        self.norm = nn.LayerNorm(cfg.d_model)
        self.q_levels = QUANTILES
        self.head = nn.Linear(cfg.d_model, P * len(STD_LEVELS_LIST))

    # ------------------------------------------------------------------ core
    def _tail_feat(self, BV, k, device, dtype):
        P = self.cfg.patch
        v = torch.zeros(BV, k, P, device=device, dtype=dtype)
        m = torch.ones(BV, k, P, device=device, dtype=dtype)
        ll = torch.zeros(BV, k, 1, device=device, dtype=dtype)
        return torch.cat([v, m, ll], dim=-1)

    def _forward_tokens(self, z_in, m_ts, k, tok_mask):
        """z_in, m_ts [BV, ctx]; k tail patches; tok_mask [BV, n_dec] (1=masked).
        Returns z-space quantiles [BV, n_dec, P, Q] for the decode region
        (fine band + tail)."""
        cfg, P = self.cfg, self.cfg.patch
        BV = z_in.shape[0]
        feat, pos = self.tokenizer(z_in, m_ts)
        tail = self._tail_feat(BV, k, z_in.device, feat.dtype)
        feat = torch.cat([feat, tail], 1)
        tail_pos = (cfg.ctx_span + torch.arange(k, device=z_in.device).float() * P) / P
        if pos.dim() == 1:
            pos = torch.cat([pos, tail_pos])
        else:
            pos = torch.cat([pos, tail_pos[None].expand(BV, -1)], 1)
        x = self.embed(feat)
        N = x.shape[1]; n_dec = cfg.zone_tokens + k
        # add learned mask token at masked decode positions
        full_mask = torch.zeros(BV, N, device=x.device, dtype=x.dtype)
        full_mask[:, -n_dec:] = tok_mask.to(x.dtype)
        x = x + full_mask[..., None] * self.mask_tok
        ang = rope_angles(pos, cfg.d_model // cfg.n_heads)
        V = getattr(self, "_V", 1)
        for blk in self.blocks:
            x = blk(x, ang, V, N)
        h = self.norm(x[:, -n_dec:])
        o = self.head(h).view(BV, n_dec, P, len(STD_LEVELS_LIST))
        return o

    # ------------------------------------------------------------------ train
    def forward_train(self, x, k, mask_patches):
        """x [B,V, ctx+k*P] raw; mask_patches [B,V, zone+k] bool (1=masked).
        Returns (loss, z-quantiles, z-targets, flat mask)."""
        cfg, P = self.cfg, self.cfg.patch
        B, V, L = x.shape
        self._V = V
        BV = B * V
        xf = x.reshape(BV, L)
        mp = mask_patches.reshape(BV, -1)                     # [BV, zone+k]
        n_dec = cfg.zone_tokens + k
        assert mp.shape[1] == n_dec
        # per-timestep mask over the decode region (last fine_span + k*P steps)
        m_ts = torch.zeros_like(xf)
        dec_span = cfg.fine_span + k * P
        m_dec = mp[..., None].expand(BV, n_dec, P).reshape(BV, dec_span).float()
        m_ts[:, -dec_span:] = m_dec
        sc = RobustScaler(xf[:, :cfg.ctx_span], m_ts[:, :cfg.ctx_span])
        z_true = sc.encode(xf)
        z_in = z_true[:, :cfg.ctx_span] * (1 - m_ts[:, :cfg.ctx_span])
        o = self._forward_tokens(z_in, m_ts[:, :cfg.ctx_span], k, mp)
        # targets: z-space patches over the decode region
        tgt = z_true[:, -dec_span:].reshape(BV, n_dec, P)
        q = self.q_levels.to(o.device)
        err = tgt[..., None] - o                              # [BV, n_dec, P, Q]
        pin = err * (q - (err < 0).float())
        w = mp.float()[..., None, None]
        loss = (pin * w).sum() / (w.sum() * P * len(STD_LEVELS_LIST) + 1e-8)
        return loss, o, tgt, mp

    # ------------------------------------------------------------------ infer
    @torch.inference_mode()
    def predict(self, ctx, k, clamp=True, valid=None):
        """ctx [B,V,C] raw (any C); returns real-space quantiles [B,V,k*P,Q].
        `valid` [B,V,C] optionally marks real (1) vs padded (0) steps when the
        caller pre-pads ragged series to a common length; padded steps are
        excluded from the scaler."""
        cfg, P = self.cfg, self.cfg.patch
        B, V, C = ctx.shape
        self._V = V
        BV = B * V
        x = ctx.reshape(BV, C)
        vld = torch.ones_like(x) if valid is None else valid.reshape(BV, C).float()
        if C < cfg.ctx_span:                                  # left-pad, edge value
            first = torch.argmax(vld, dim=1, keepdim=True)    # first real step
            edge = torch.gather(x, 1, first)
            pad = edge.expand(BV, cfg.ctx_span - C)
            xf = torch.cat([pad, x], 1)
            valid_f = torch.cat([torch.zeros(BV, cfg.ctx_span - C, device=x.device), vld], 1)
        else:
            xf = x[:, -cfg.ctx_span:]
            valid_f = vld[:, -cfg.ctx_span:]
        # padded/invalid steps carry the first-real (edge) value, mask-channel 0
        first = torch.argmax(valid_f, dim=1, keepdim=True)
        edge = torch.gather(xf, 1, first)
        xf = torch.where(valid_f > 0.5, xf, edge)
        valid = valid_f
        sc = RobustScaler(xf, m=(1 - valid))                  # scaler on real steps only
        z_in = sc.encode(xf)                                  # padded steps: edge value, mask=0
        m_ts = torch.zeros_like(z_in)
        tok_mask = torch.zeros(BV, cfg.zone_tokens + k, device=x.device)
        tok_mask[:, -k:] = 1.0
        o = self._forward_tokens(z_in, m_ts, k, tok_mask)     # [BV, zone+k, P, Q]
        o = o[:, -k:].reshape(BV, k*P, len(STD_LEVELS_LIST))
        o, _ = torch.sort(o, dim=-1)                          # de-cross
        q = (torch.sinh(o) * sc.scale[..., None] + sc.loc[..., None]).reshape(B, V, k*P, -1)
        if clamp:                                             # wide sanity clamp
            lo = ctx.min(dim=-1, keepdim=True).values[..., None]
            hi = ctx.max(dim=-1, keepdim=True).values[..., None]
            rng = (hi - lo).clamp_min(1e-3)
            q = q.clamp(lo - 10*rng, hi + 10*rng)
        return q

    @torch.inference_mode()
    def rollout(self, ctx, total_h, block_k=None):
        """Median-feedback block decoding past the trained tail length."""
        cfg, P = self.cfg, self.cfg.patch
        block_k = block_k or cfg.train_kmax
        cur, preds = ctx, []
        n_blocks = math.ceil(total_h / (block_k * P))
        for _ in range(n_blocks):
            q = self.predict(cur, block_k)                    # [B,V,bk*P,Q]
            med = q[..., q.shape[-1] // 2]
            preds.append(med)
            cur = torch.cat([cur, med], dim=-1)[..., -cfg.ctx_span:]
        return torch.cat(preds, dim=-1)[..., :total_h]

def count_params(model):
    return sum(p.numel() for p in model.parameters())

# --------------------------------------------------------------------------- CPM sampler
def sample_cpm_mask(cfg, B, V, rng):
    """(k, mask_patches [B,V,zone+k]): tail span k~U{kmin..kmax} always masked,
    plus c~U{0..cmax} interior spans in the fine band, total <= pmax fraction."""
    k = int(rng.integers(cfg.train_kmin, cfg.train_kmax + 1))
    n_dec = cfg.zone_tokens + k
    mp = torch.zeros(B, V, n_dec, dtype=torch.bool)
    mp[..., -k:] = True
    budget = max(0, int(cfg.cpm_pmax * n_dec) - k)
    for b in range(B):
        for v in range(V):
            used, c = 0, int(rng.integers(0, cfg.cpm_cmax + 1))
            for _ in range(c):
                if used >= budget: break
                w = int(rng.integers(1, cfg.cpm_span_max + 1))
                w = min(w, budget - used)
                s = int(rng.integers(0, cfg.zone_tokens - w + 1))
                mp[b, v, s:s+w] = True
                used += w
    return k, mp


In [ ]:
# ============================================================================
# OPTIMIZER  ·  NorMuon (transformer matrices) + AdamW (embed/head/bias/norm),
# mirroring the Toto-2 split. No u-muP here, so LRs are plain-parametrization
# values calibrated by the optional LR_SWEEP cell below and shared across arms —
# widths are identical in every arm, so approximate transfer holds.
# WSD schedule: warmup -> stable -> linear decay tail. Note: the 15k
# rank-stability checkpoints are UNdecayed for every arm (paired comparison
# stays valid); only the 30k finals see the decay tail.
# ============================================================================
import torch, math

def newton_schulz(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float(); X = X / (X.norm() + eps)
    transposed = X.shape[0] > X.shape[1]
    if transposed: X = X.T
    for _ in range(steps):
        A = X @ X.T
        X = a*X + (b*A + c*A@A) @ X
    return X.T if transposed else X

class NorMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=8e-4, momentum=0.95, beta2=0.999, eps=1e-8):
        super().__init__(list(params), dict(lr=lr, momentum=momentum, beta2=beta2, eps=eps))
    @torch.no_grad()
    def step(self):
        for g in self.param_groups:
            for p in g["params"]:
                if p.grad is None: continue
                st = self.state[p]
                if "buf" not in st:
                    st["buf"] = torch.zeros_like(p.grad)
                    st["v"] = torch.zeros(p.grad.shape[0], device=p.device)
                buf = st["buf"]; buf.mul_(g["momentum"]).add_(p.grad)
                O = newton_schulz(buf)
                st["v"].mul_(g["beta2"]).add_((O*O).mean(1), alpha=1-g["beta2"])
                p.add_(O / (st["v"].sqrt()[:, None] + g["eps"]), alpha=-g["lr"])

def build_optimizers(model, cfg):
    mat, other = [], []
    for n, p in model.named_parameters():
        in_backbone = n.startswith("blocks") and p.ndim == 2
        (mat if in_backbone else other).append(p)
    return [NorMuon(mat, lr=cfg.normuon_lr),
            torch.optim.AdamW(other, lr=cfg.lr, weight_decay=cfg.weight_decay,
                              betas=(0.9, 0.98))]

def lr_scale(step, total, warmup_frac, decay_frac):
    w = max(1, int(total * warmup_frac)); d = max(1, int(total * decay_frac))
    if step < w:            return step / w
    if step > total - d:    return max(0.0, (total - step) / d)
    return 1.0

def set_lr(opts, base_lrs, scale):
    for opt, base in zip(opts, base_lrs):
        for g in opt.param_groups: g["lr"] = base * scale


## 4 · Training
Resumable per-run training with paired data + paired CPM masks across arms, synthetic val curve, 15k/30k snapshots.

In [ ]:
# ============================================================================
# TRAINING  ·  one run = one Cfg. Resumable from Drive; bf16 autocast on
# Ampere+; deterministic paired data via CorpusSampler(step). Every run logs
# a scale-normalized synthetic val CRPS curve; gift_ckpts (15k/30k) snapshot
# weights for the rank-stability check.
# ============================================================================
import torch, numpy as np, os, json, time, random

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = (torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported())
             else None)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def make_val_batch(cfg, n=64, seed=777):
    """Fixed held-out synthetic batch from the SAME ensemble prior (never trained on:
    val seed is outside the corpus shard seed range)."""
    rng = np.random.default_rng(seed)
    L = cfg.ctx_span + cfg.train_kmax * cfg.patch
    x = sample_ensemble(n, L, rng)
    return torch.from_numpy(x).unsqueeze(1)                    # [n, 1, L]

@torch.inference_mode()
def val_crps(model, cfg, vb):
    """Scale-normalized CRPS at the full trained tail (kmax patches)."""
    model.eval()
    k = cfg.train_kmax
    ctx = vb[..., :cfg.ctx_span].to(DEVICE)
    tgt = vb[..., cfg.ctx_span:cfg.ctx_span + k*cfg.patch].to(DEVICE)
    q = model.predict(ctx, k)                                  # [n,1,kP,9]
    lv = model.q_levels.to(DEVICE)
    err = tgt[..., None] - q
    pin = 2.0 * err * (lv - (err < 0).float())
    per = pin.mean(dim=(-1, -2))                               # [n,1]
    scale = RobustScaler(ctx.reshape(ctx.shape[0], -1)).scale.clamp_min(1e-6)
    model.train()
    return float((per.squeeze(1) / scale.squeeze(1)).mean())

def _wb(d, step=None):
    """Log to the active W&B run if there is one; silent no-op otherwise."""
    try:
        import wandb
        if wandb.run is not None:
            wandb.log(d, step=step)
    except Exception:
        pass

def run_dir(cfg):
    d = os.path.join(CKPT_DIR, "v10_runs", cfg.run_name)
    os.makedirs(d, exist_ok=True)
    return d

def train_one(cfg, verbose=True):
    set_seed(cfg.seed)
    model = MiniTSFM2(cfg).to(DEVICE)
    opts = build_optimizers(model, cfg)
    base_lrs = [cfg.normuon_lr, cfg.lr]
    sampler = CorpusSampler(cfg.pool, cfg.data_seed)
    vb = make_val_batch(cfg)
    d = run_dir(cfg)
    log = {"step": [], "val_crps": [], "train_loss": []}
    start = 0
    # ---- resume -------------------------------------------------------------
    latest = os.path.join(d, "latest.pt")
    if os.path.exists(latest):
        ck = torch.load(latest, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); start = ck["step"]
        for o, s in zip(opts, ck["opts"]): o.load_state_dict(s)
        log = ck.get("log", log)
        print(f"[{cfg.run_name}] resumed at step {start}")
    if start >= cfg.steps:
        print(f"[{cfg.run_name}] already complete"); return model, log
    if verbose:
        print(f"[{cfg.run_name}] params={count_params(model)/1e6:.2f}M "
              f"tokens/seq={cfg.n_ctx_tokens}+tail device={DEVICE} amp={AMP_DTYPE}")
    mask_rng = np.random.default_rng(cfg.data_seed + 50_000)   # paired masking too
    t0, model_train_loss = time.time(), 0.0
    model.train()
    for step in range(start, cfg.steps):
        x = torch.from_numpy(sample_series(sampler, step, cfg)).to(DEVICE, non_blocking=True)
        k, mp = sample_cpm_mask(cfg, cfg.batch, cfg.n_variates, mask_rng)
        mp = mp.to(DEVICE)
        set_lr(opts, base_lrs, lr_scale(step, cfg.steps, cfg.warmup_frac, cfg.decay_frac))
        for o in opts: o.zero_grad(set_to_none=True)
        if AMP_DTYPE is not None:
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss, *_ = model.forward_train(x, k, mp)
        else:
            loss, *_ = model.forward_train(x, k, mp)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        for o in opts: o.step()
        model_train_loss = 0.98*model_train_loss + 0.02*float(loss.detach()) if step > start else float(loss.detach())
        # ---- eval / logging ----------------------------------------------------
        if (step + 1) % cfg.eval_every == 0 or step + 1 == cfg.steps:
            vc = val_crps(model, cfg, vb)
            log["step"].append(step + 1); log["val_crps"].append(vc)
            log["train_loss"].append(model_train_loss)
            _wb({"train_loss": model_train_loss, "val_crps": vc,
                 "lr_scale": lr_scale(step, cfg.steps, cfg.warmup_frac, cfg.decay_frac)},
                step=step + 1)
            if verbose:
                sps = (step + 1 - start) / max(1e-9, time.time() - t0)
                print(f"[{cfg.run_name}] step {step+1}/{cfg.steps} "
                      f"loss={model_train_loss:.4f} valCRPS={vc:.4f} ({sps:.1f} it/s)")
        # ---- checkpoints ---------------------------------------------------------
        if (step + 1) % cfg.ckpt_every == 0 or step + 1 == cfg.steps:
            torch.save({"model": model.state_dict(), "step": step + 1,
                        "opts": [o.state_dict() for o in opts],
                        "cfg": asdict(cfg), "log": log}, latest)
        if (step + 1) in cfg.gift_ckpts:
            torch.save({"model": model.state_dict(), "step": step + 1,
                        "cfg": asdict(cfg)}, os.path.join(d, f"ckpt_{step+1}.pt"))
    with open(os.path.join(d, "log.json"), "w") as f:
        json.dump({"cfg": asdict(cfg), "log": log}, f)
    return model, log


In [ ]:
# ============================================================================
# OPTIONAL: LR CALIBRATION on T0  ·  run once BEFORE the matrix (set True).
# No u-muP here, so the Toto-2 LRs don't transfer; this 3-point sweep on the
# control arm (~25 min total on A100 at 2k steps/point) picks the multiplier.
# All arms share identical widths/shapes, so the T0-calibrated LRs transfer
# across arms to good approximation. Apply the winner by editing Cfg defaults
# (lr, normuon_lr) before running the matrix.
# ============================================================================
LR_SWEEP = False
if LR_SWEEP:
    sweep = {}
    for mult in (0.5, 1.0, 2.0):
        cfg = make_arm("T0", 0)
        cfg.steps, cfg.eval_every = 2_000, 500
        cfg.ckpt_every, cfg.gift_ckpts = 10**9, ()
        cfg.lr *= mult; cfg.normuon_lr *= mult
        cfg.run_name = f"lrsweep_x{mult}"
        _, log = train_one(cfg)
        sweep[mult] = log["val_crps"][-1]
        print(f"lr multiplier x{mult}: final val CRPS = {sweep[mult]:.4f}")
    best = min(sweep, key=sweep.get)
    print(f"\nbest multiplier: x{best} -> set Cfg.lr={1e-3*best:g}, "
          f"Cfg.normuon_lr={8e-4*best:g}, then run the matrix.")
    if best != 1.0 and min(sweep[0.5], sweep[2.0]) < sweep[1.0]:
        print("(edge of grid won — consider extending the sweep one step further)")


## 5 · Evaluation
Fixed 14-task GIFT-Eval dev subset (rest held out for the promoted winner) + Toto-2-Fig-10-style 8,192-step rollout probe.

In [ ]:
# ============================================================================
# GIFT-EVAL  ·  dev-subset harness (official evaluate_model path, CRPS ==
# leaderboard mean_weighted_sum_quantile_loss). DEV_SETS is the FIXED design
# subset — do NOT grow it while iterating on tokenizers, or you Goodhart the
# choice; the remaining ~85 tasks stay held out for the promoted winner.
# Unresolvable names are skipped with a warning (GIFT names occasionally
# shift between releases), so the harness degrades gracefully.
# ============================================================================
import os, numpy as np, torch

QL = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# (name, term) — stratified across frequency; the 10T/15T/H entries form the
# "long-seasonality" slice where the iso-token pyramid (T2) should show its
# advantage if the long-history hypothesis is real.
DEV_SETS = [
    ("m4_weekly", "short"), ("m4_daily", "short"), ("m4_hourly", "short"),
    ("m4_monthly", "short"),
    ("electricity/15T", "short"), ("electricity/H", "short"),
    ("solar/10T", "short"), ("solar/H", "short"),
    ("ett1/15T", "short"), ("ett2/H", "short"),
    ("jena_weather/10T", "short"),
    ("us_births/D", "short"), ("covid_deaths", "short"), ("hospital", "short"),
]
LONG_SEASONALITY_FREQS = {"10T", "15T", "H", "h", "10min", "15min"}

def _download_dev_sets():
    from huggingface_hub import snapshot_download
    pats = sorted({n.split("/")[0] for n, _ in DEV_SETS})
    snapshot_download("Salesforce/GiftEval", repo_type="dataset",
                      local_dir=GIFT_EVAL_DIR, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=[f"{p}/*" for p in pats] + [f"{p}*" for p in pats])
    os.environ["GIFT_EVAL"] = GIFT_EVAL_DIR

class TSFMPredictor:
    """gluonts RepresentablePredictor around MiniTSFM2.predict (batched,
    ragged-aware via the valid mask; NaNs forward-filled)."""
    def __init__(self, model, H, batch_size=256):
        from gluonts.model.predictor import RepresentablePredictor
        self.model, self.H, self.bs = model, H, batch_size
        self.prediction_length = H
        P = model.cfg.patch
        self.k = math.ceil(H / P)
    def predict(self, dataset, **kwargs):
        from gluonts.model.forecast import QuantileForecast
        entries = list(dataset)
        for i in range(0, len(entries), self.bs):
            chunk = entries[i:i+self.bs]
            Cmax = min(self.model.cfg.ctx_span,
                       max(len(np.atleast_1d(e["target"])) for e in chunk))
            Cmax = max(Cmax, self.model.cfg.patch)
            ctx = np.zeros((len(chunk), 1, Cmax), np.float32)
            val = np.zeros((len(chunk), 1, Cmax), np.float32)
            for j, e in enumerate(chunk):
                y = np.asarray(e["target"], np.float32).reshape(-1)
                # forward-fill NaNs, then zero any leading NaNs
                idx = np.where(~np.isnan(y), np.arange(len(y)), 0)
                np.maximum.accumulate(idx, out=idx)
                y = np.nan_to_num(y[idx], nan=0.0)
                y = y[-Cmax:]
                ctx[j, 0, -len(y):] = y
                val[j, 0, -len(y):] = 1.0
            with torch.autocast("cuda", dtype=torch.bfloat16,
                                enabled=(DEVICE == "cuda" and AMP_DTYPE is not None)):
                if self.k <= self.model.cfg.train_kmax:
                    q = self.model.predict(torch.from_numpy(ctx).to(DEVICE), self.k,
                                           valid=torch.from_numpy(val).to(DEVICE))
                else:  # horizon beyond trained tail: block rollout, per-block quantiles
                    q = self._rollout_quantiles(torch.from_numpy(ctx).to(DEVICE),
                                                torch.from_numpy(val).to(DEVICE))
            q = q.float().cpu().numpy()[:, 0, :self.H, :]      # [n, H, 9]
            for j, e in enumerate(chunk):
                yield QuantileForecast(
                    forecast_arrays=q[j].T, forecast_keys=[str(p) for p in QL],
                    start_date=e["start"] + len(np.atleast_1d(e["target"])),
                    item_id=e.get("item_id"))
    def _rollout_quantiles(self, ctx, val):
        m, k_max = self.model, self.model.cfg.train_kmax
        outs, cur, cv, k_left = [], ctx, val, self.k
        while k_left > 0:
            kb = min(k_left, k_max)
            q = m.predict(cur, kb, valid=cv)                  # [B,1,kb*P,9]
            outs.append(q)
            med = q[..., q.shape[-1] // 2]
            cur = torch.cat([cur, med], dim=-1)[..., -m.cfg.ctx_span:]
            cv = torch.cat([cv, torch.ones_like(med)], dim=-1)[..., -m.cfg.ctx_span:]
            k_left -= kb
        return torch.cat(outs, dim=2)

def eval_gift_dev(model, tag=""):
    """Returns {dataset: {CRPS, MASE, freq}} + geometric-mean CRPS + slices."""
    from gluonts.model import evaluate_model
    from gluonts.time_feature import get_seasonality
    from gluonts.ev.metrics import MASE, MeanWeightedSumQuantileLoss
    from gift_eval.data import Dataset as GEDataset
    _download_dev_sets()
    model.eval()
    rows = {}
    for name, term in DEV_SETS:
        try:
            to_uni = GEDataset(name=name, term=term).target_dim != 1
            ds = GEDataset(name=name, term=term, to_univariate=to_uni)
            season = get_seasonality(ds.freq)
            res = evaluate_model(
                TSFMPredictor(model, ds.prediction_length), test_data=ds.test_data,
                metrics=[MASE(), MeanWeightedSumQuantileLoss(quantile_levels=QL)],
                batch_size=512, axis=None, mask_invalid_label=True,
                allow_nan_forecast=False, seasonality=season)
            rows[name] = {"CRPS": float(res["mean_weighted_sum_quantile_loss"].iloc[0]),
                          "MASE": float(res["MASE[0.5]"].iloc[0]), "freq": ds.freq}
            print(f"  {tag} {name:22s} CRPS={rows[name]['CRPS']:.4f} MASE={rows[name]['MASE']:.3f}")
        except Exception as ex:
            print(f"  {tag} {name:22s} SKIPPED ({type(ex).__name__}: {ex})")
    crps = [r["CRPS"] for r in rows.values() if np.isfinite(r["CRPS"]) and r["CRPS"] > 0]
    out = {"per_dataset": rows,
           "gm_crps": float(np.exp(np.mean(np.log(crps)))) if crps else float("nan")}
    ls = [r["CRPS"] for r in rows.values()
          if r["freq"] in LONG_SEASONALITY_FREQS and np.isfinite(r["CRPS"]) and r["CRPS"] > 0]
    ss = [r["CRPS"] for r in rows.values()
          if r["freq"] not in LONG_SEASONALITY_FREQS and np.isfinite(r["CRPS"]) and r["CRPS"] > 0]
    out["gm_crps_long_season"] = float(np.exp(np.mean(np.log(ls)))) if ls else float("nan")
    out["gm_crps_other"] = float(np.exp(np.mean(np.log(ss)))) if ss else float("nan")
    print(f"{tag} GM-CRPS={out['gm_crps']:.4f}  long-season={out['gm_crps_long_season']:.4f} "
          f" other={out['gm_crps_other']:.4f}  ({len(rows)}/{len(DEV_SETS)} datasets)")
    return out


In [ ]:
# ============================================================================
# PROBES + AGGREGATION  ·  (a) Toto-2-Fig-10-style long-horizon stability on
# the superimposed periods-(500,100,20) signal: 8192-step block rollout,
# per-bucket Pearson r + amplitude retention. (b) Cross-run aggregate table
# with the tripwire checks (seed noise floor, 15k-vs-30k rank stability).
# ============================================================================
import numpy as np, torch, os, json, glob

def make_multiscale(n, rng, periods=(500, 100, 20), noise=0.05):
    t = np.arange(n)
    x = np.zeros(n)
    for p in periods:
        x += rng.uniform(0.5, 1.5) * np.sin(2*np.pi*t/p + rng.uniform(0, 2*np.pi))
    return (x + noise * rng.standard_normal(n)).astype(np.float32)

@torch.inference_mode()
def long_horizon_probe(model, n_series=16, total_h=8192, n_buckets=8, seed=123):
    """Block-rollout 8192 steps past a full-context window of the multiscale
    signal; report Pearson r and amplitude retention (pred_std/true_std) per
    horizon bucket. Collapse -> r falls and amp -> 0 in late buckets."""
    cfg = model.cfg
    rng = np.random.default_rng(seed)
    L = cfg.ctx_span + total_h
    xs = np.stack([make_multiscale(L, rng) for _ in range(n_series)])
    x = torch.from_numpy(xs).unsqueeze(1).to(DEVICE)              # [n,1,L]
    ctx, tgt = x[..., :cfg.ctx_span], x[..., cfg.ctx_span:]
    pred = model.rollout(ctx, total_h)                            # [n,1,H] median
    bs = total_h // n_buckets
    pearson, amp = [], []
    for i in range(n_buckets):
        p = pred[..., i*bs:(i+1)*bs].reshape(-1).float().cpu().numpy()
        t = tgt[..., i*bs:(i+1)*bs].reshape(-1).float().cpu().numpy()
        pc = float(np.corrcoef(p, t)[0, 1]) if p.std() > 1e-9 else 0.0
        pearson.append(round(pc, 3))
        amp.append(round(float(p.std() / (t.std() + 1e-9)), 3))
    return {"pearson": pearson, "amp_retention": amp}

# ---------------------------------------------------------------------------- aggregation
def load_results():
    rows = []
    for p in glob.glob(os.path.join(CKPT_DIR, "v10_runs", "*", "results.json")):
        with open(p) as f:
            rows.append(json.load(f))
    return rows

def aggregate_table(rows):
    import pandas as pd
    if not rows:
        print("no results yet"); return None
    df = pd.DataFrame([{
        "arm": r["run_name"].rsplit("_s", 1)[0], "seed": r["cfg"]["seed"],
        "gm_crps_30k": r.get("gift_final", {}).get("gm_crps"),
        "gm_crps_15k": r.get("gift_15k", {}).get("gm_crps"),
        "long_season_30k": r.get("gift_final", {}).get("gm_crps_long_season"),
        "other_30k": r.get("gift_final", {}).get("gm_crps_other"),
        "probe_r_last": (r.get("probe", {}).get("pearson") or [None])[-1],
        "probe_amp_last": (r.get("probe", {}).get("amp_retention") or [None])[-1],
    } for r in rows])
    agg = df.groupby("arm").agg(
        gm_crps_mean=("gm_crps_30k", "mean"), gm_crps_std=("gm_crps_30k", "std"),
        gm_crps_15k=("gm_crps_15k", "mean"),
        long_season=("long_season_30k", "mean"), other=("other_30k", "mean"),
        probe_r=("probe_r_last", "mean"), probe_amp=("probe_amp_last", "mean"),
        n=("seed", "count")).sort_values("gm_crps_mean")
    print(agg.round(4).to_string())

    # ---- tripwires ----------------------------------------------------------
    print("\n--- tripwire checks ---")
    arms = agg.index.tolist()
    sig = float(np.nanmax(agg["gm_crps_std"].values)) if len(agg) else float("nan")
    print(f"seed noise floor (max arm sigma): {sig:.4f}")
    for i in range(len(arms) - 1):
        gap = agg["gm_crps_mean"].iloc[i+1] - agg["gm_crps_mean"].iloc[i]
        if not np.isfinite(sig):
            ok = "sigma unknown (single seed) — run more seeds before concluding"
        else:
            ok = "RESOLVED" if gap > 2 * sig else "UNRESOLVED (< 2*sigma) — add seeds/steps"
        print(f"  {arms[i]} vs {arms[i+1]}: gap={gap:.4f} -> {ok}")
    r15 = agg.sort_values("gm_crps_15k").index.tolist()
    r30 = agg.sort_values("gm_crps_mean").index.tolist()
    if r15[:2] != r30[:2]:
        print(f"  RANK FLIP among top-2 between 15k {r15[:2]} and 30k {r30[:2]} "
              f"-> extend to 60k before promoting")
    else:
        print(f"  rank stable 15k->30k: {r30}")
    # T1 compression-tax gate
    if {"T0_fixed", "T1_pyr_ctx"} <= set(agg.index):
        tax = (agg.loc["T1_pyr_ctx", "gm_crps_mean"] / agg.loc["T0_fixed", "gm_crps_mean"] - 1)
        print(f"  H1 compression tax (T1 vs T0): {100*tax:+.2f}% "
              f"{'(OK, <=2%)' if tax <= 0.02 else '(FAILS H1 gate — read T2 as gain minus tax)'}")
    return agg


## 6 · Hugging Face Hub push
Per-run: weights (final + 15k), config, metrics; plus an auto-updated experiment card. Needs a WRITE token in the `HF_TOKEN` Colab secret.

In [ ]:
# ============================================================================
# HUGGING FACE HUB  ·  push each run's final weights + config + metrics + a
# model card to a single repo (subfolder per run). Needs a WRITE token in
# HF_TOKEN. Uploads are best-effort: failures never kill the run matrix.
# Repo layout:
#   README.md                      <- experiment-level card (auto-updated)
#   T0_fixed_s0/model.pt           <- final (30k, decayed) weights
#   T0_fixed_s0/ckpt_15000.pt      <- rank-stability snapshot
#   T0_fixed_s0/config.json
#   T0_fixed_s0/results.json
# Reload later with:
#   sd = torch.load(hf_hub_download(REPO, "T0_fixed_s0/model.pt"))["model"]
#   cfg = Cfg(**json.load(open(hf_hub_download(REPO, "T0_fixed_s0/config.json"))))
#   m = MiniTSFM2(cfg); m.load_state_dict(sd)
# ============================================================================
import os, json

def _hf_api():
    from huggingface_hub import HfApi
    tok = os.environ.get("HF_TOKEN") or None
    api = HfApi(token=tok)
    return api, api.whoami()["name"]

def hf_repo_id():
    global HF_REPO
    if HF_REPO:
        return HF_REPO
    _, user = _hf_api()
    HF_REPO = f"{user}/tsfm-toto2-4m-tokenizer-ablation"
    return HF_REPO

_CARD = """---
license: apache-2.0
tags: [time-series, forecasting, foundation-model, tokenizer-ablation, toto-2-recipe]
---
# TSFM tokenizer ablation — Toto-2-4m-recipe clone (v10)

Context-tokenizer ablation on a ~3.6M-param decoder-only patched transformer
trained on a TempoPFN-style synthetic prior ensemble with contiguous patch
masking, a 9-level quantile head, NorMuon+AdamW, and index RoPE at
time-scaled positions. Horizon decoding is fixed patch-32 in every arm; only
the CONTEXT tokenizer varies:

| arm | tokenizer | history | ctx tokens |
|-----|-----------|---------|-----------|
| T0  | fixed-32 (control) | 4,096 | 128 |
| T1  | pyramid, iso-context | 4,096 | 44 |
| T2  | pyramid, iso-token | 16,384 | 128 |
| T3  | adaptive equal-surprise | 16,384 | 128 |

Each subfolder is one (arm, seed) run: `model.pt` (final), `ckpt_15000.pt`
(rank-stability snapshot), `config.json`, `results.json` (dev-GIFT CRPS +
long-horizon probe). Dev metrics use a fixed 14-task GIFT-Eval subset — NOT
the full leaderboard; treat numbers as ablation-internal, not comparable to
published GIFT scores. Generated by the v10 experiment notebook.

## Results
{RESULTS}
"""

def push_run(cfg, results=None):
    if not HF_PUSH:
        return
    try:
        api, _ = _hf_api()
        repo = hf_repo_id()
        api.create_repo(repo, exist_ok=True, private=False)
        d = run_dir(cfg)
        ops = []
        latest = os.path.join(d, "latest.pt")
        if os.path.exists(latest):
            ops.append((latest, f"{cfg.run_name}/model.pt"))
        for s in cfg.gift_ckpts[:-1]:
            p = os.path.join(d, f"ckpt_{s}.pt")
            if os.path.exists(p):
                ops.append((p, f"{cfg.run_name}/ckpt_{s}.pt"))
        cfg_path = os.path.join(d, "config.json")
        with open(cfg_path, "w") as f:
            json.dump(asdict(cfg), f, indent=2)
        ops.append((cfg_path, f"{cfg.run_name}/config.json"))
        res_path = os.path.join(d, "results.json")
        if results is not None:
            with open(res_path, "w") as f:
                json.dump(results, f, indent=2)
        if os.path.exists(res_path):
            ops.append((res_path, f"{cfg.run_name}/results.json"))
        for local, remote in ops:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=repo)
        print(f"[hf] pushed {cfg.run_name} -> https://huggingface.co/{repo}")
    except Exception as ex:
        print(f"[hf] push skipped for {cfg.run_name}: {type(ex).__name__}: {ex}")

def push_experiment_card():
    """Refresh the repo-level README with the current aggregate table."""
    if not HF_PUSH:
        return
    try:
        import io, contextlib
        api, _ = _hf_api(); repo = hf_repo_id()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            aggregate_table(load_results())
        card = _CARD.replace("{RESULTS}", "```\n" + buf.getvalue() + "\n```")
        api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
                        repo_id=repo)
        print(f"[hf] card updated -> https://huggingface.co/{repo}")
    except Exception as ex:
        print(f"[hf] card update skipped: {type(ex).__name__}: {ex}")

if HF_PUSH and not os.environ.get("HF_TOKEN"):
    print("NOTE: HF_PUSH=True but no HF_TOKEN found — uploads will be skipped.\n"
          "Add a WRITE token as a Colab secret named HF_TOKEN to enable them.")


## 7 · RUN MATRIX (the long cell)
4 arms × 3 seeds × 30k steps. Fully resumable; each run pushes to the Hub when done. Set `SMOKE_MODE=True` for an end-to-end pipeline check first.

In [ ]:
# ============================================================================
# RUN MATRIX  ·  4 arms x 3 seeds x 30k steps (~2 A100-h per run at the
# default batch 64 x 8 variates; ~1 A100-day total). Fully resumable: rerun
# this cell after any disconnect and it continues where it stopped. Each run:
# train -> dev-GIFT at final + 15k ckpt -> long-horizon probe -> Drive
# results.json -> HF push. Set SMOKE_MODE=True first to sanity-check the whole
# pipeline end-to-end in a few minutes (tiny steps, 2 dev datasets, 1 seed).
# ============================================================================
import os, json, gc, torch

SMOKE_MODE = False
EVAL_15K = True            # rank-stability GIFT pass on the 15k checkpoint

runs = RUNS
if SMOKE_MODE:
    global DEV_SETS
    DEV_SETS = DEV_SETS[:2]
    runs = [make_arm(a, 0) for a in ("T0", "T1", "T2", "T3")]
    for c in runs:
        c.steps, c.eval_every, c.ckpt_every = 200, 50, 100
        c.gift_ckpts = (100, 200)
        c.batch, c.n_variates = 8, 2
        c.run_name += "_smoke"

for cfg in runs:
    d = run_dir(cfg)
    res_path = os.path.join(d, "results.json")
    if os.path.exists(res_path) and not SMOKE_MODE:
        print(f"[skip] {cfg.run_name} — results.json exists")
        continue
    print("=" * 70); print("RUN", cfg.run_name)
    wb = None
    if WANDB_ENABLE:
        try:
            import wandb
            wb = wandb.init(project=WANDB_PROJECT, name=cfg.run_name,
                            id=cfg.run_name, resume="allow",
                            config=asdict(cfg), reinit=True)
        except Exception as ex:
            print(f"[wandb] disabled for this run ({type(ex).__name__}: {ex})")
    model, log = train_one(cfg)
    results = {"run_name": cfg.run_name, "cfg": asdict(cfg), "val_log": log}
    # ---- dev GIFT: final (30k, decayed) --------------------------------------
    results["gift_final"] = eval_gift_dev(model, tag=f"{cfg.run_name}@final")
    # ---- dev GIFT: 15k snapshot (undecayed; paired across arms) ---------------
    ck15 = os.path.join(d, f"ckpt_{cfg.gift_ckpts[0]}.pt")
    if EVAL_15K and os.path.exists(ck15):
        m15 = MiniTSFM2(cfg).to(DEVICE)
        m15.load_state_dict(torch.load(ck15, map_location=DEVICE,
                                       weights_only=False)["model"])
        results["gift_15k"] = eval_gift_dev(m15, tag=f"{cfg.run_name}@15k")
        del m15
    # ---- long-horizon probe ----------------------------------------------------
    results["probe"] = long_horizon_probe(
        model, total_h=(512 if SMOKE_MODE else 8192))
    print(f"probe pearson per bucket : {results['probe']['pearson']}")
    print(f"probe amp retention      : {results['probe']['amp_retention']}")
    # ---- persist + push ----------------------------------------------------------
    with open(res_path, "w") as f:
        json.dump(results, f, indent=2)
    push_run(cfg, results)
    if wb is not None:
        try:
            wb.summary.update({
                "gm_crps_final": results["gift_final"]["gm_crps"],
                "gm_crps_long_season": results["gift_final"]["gm_crps_long_season"],
                "gm_crps_other": results["gift_final"]["gm_crps_other"],
                "gm_crps_15k": results.get("gift_15k", {}).get("gm_crps"),
                "probe_pearson_last": results["probe"]["pearson"][-1],
                "probe_amp_last": results["probe"]["amp_retention"][-1]})
            wb.finish()
        except Exception:
            pass
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

push_experiment_card()
print("\nRUN MATRIX COMPLETE")


## 8 · Results
Aggregate table + tripwires (seed noise floor, 15k↔30k rank stability, H1 compression-tax gate) + plots. Rerunnable anytime — reads Drive.

In [ ]:
# ============================================================================
# PLOTS  ·  (1) synthetic val-CRPS curves per arm (seeds as thin lines),
# (2) long-horizon probe: Pearson r and amplitude retention vs horizon bucket.
# Rerunnable any time — reads results.json files from Drive.
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np

rows = load_results()
if not rows:
    print("no results yet — run the matrix first")
else:
    arms = sorted({r["run_name"].rsplit("_s", 1)[0] for r in rows})
    colors = {a: c for a, c in zip(arms, plt.cm.tab10.colors)}
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    for r in rows:
        arm = r["run_name"].rsplit("_s", 1)[0]
        lg = r.get("val_log", {})
        if lg.get("step"):
            axes[0].plot(lg["step"], lg["val_crps"], color=colors[arm], alpha=0.55,
                         label=arm if r["cfg"]["seed"] == 0 else None)
    axes[0].set(title="synthetic val CRPS (scale-normalized)", xlabel="step",
                ylabel="CRPS", yscale="log")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    for r in rows:
        arm = r["run_name"].rsplit("_s", 1)[0]
        pr = r.get("probe", {})
        if pr:
            x = np.arange(1, len(pr["pearson"]) + 1)
            axes[1].plot(x, pr["pearson"], "o-", color=colors[arm], alpha=0.55,
                         label=arm if r["cfg"]["seed"] == 0 else None)
            axes[2].plot(x, pr["amp_retention"], "o-", color=colors[arm], alpha=0.55)
    axes[1].set(title="8192-step rollout: Pearson r / bucket", xlabel="horizon bucket",
                ylim=(-0.1, 1.02))
    axes[1].axhline(0, color="k", lw=0.5); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    axes[2].set(title="amplitude retention / bucket", xlabel="horizon bucket")
    axes[2].axhline(1.0, color="k", lw=0.5, ls="--"); axes[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


## Notes & caveats (read before believing the numbers)
- **Decision rule:** promote the winner (+ T0 control) to the full 400k-step paper-scale recipe only if its gap clears 2× the seed σ **and** the 15k→30k ranking is stable. `T2 ≈ T0` at 4M params is a *null*, not a kill — capacity may bind before context does; recheck at the next scale if cheap.
- **Dev subset ≠ leaderboard.** CRPS here is the official metric on 14 tasks, but numbers are ablation-internal; don't compare to published GIFT scores.
- **15k checkpoints are undecayed** (WSD); the 15k comparison is undecayed-vs-undecayed, which is fine for rank stability but not for absolute quality.
- **Long-horizon quantiles** past the trained 32-patch tail come from per-block predictions with median feedback — uncertainty does **not** accumulate across blocks. Fine for the Pearson/amplitude probe; don't read the far-horizon intervals as calibrated.
- **Variates are independent draws**, so variate attention (last layer) has little to learn here; it's kept for architectural fidelity. Wire correlated variates into `sample_series` if that matters to you.
- **Native generator ≠ official TempoPFN.** Same prior families, different code; GP-family priors are spectral approximations. To use the official generator: install the repo, wrap it as `(B, L, rng) -> [B, L]`, assign `OFFICIAL_SAMPLER`, delete the corpus dir, rebuild.
- **Deviations from the Toto-2-4m recipe:** V=8 (not 32), no u-µP (LRs re-calibrated once on T0, shared — same widths in every arm), CPM restricted to the fine band + tail for cross-arm comparability, 30k of 400k steps (late-training effects unobserved).